# Compress ConvNeXt-Tiny (109 MB, 28 M parameters) on CIFAR-10 — Kaggle or Colab notebook (Member 3)

`ImageNet-pretrained ConvNeXt-Tiny  →  new 10-class head, fine-tuned on CIFAR-10  →  gradual prune + fine-tune  →  INT8  →  ONNX  →  verify`

**Why this model is hard to compress:** a modern CNN: 7x7 depthwise convolutions, LayerNorm and GELU instead of BatchNorm and ReLU. Both pruning and INT8 are less forgiving here than on classic ResNets.
**What to watch:** No BatchNorm, depthwise convs and LayerNorm: INT8 quantization of ConvNeXt is known to lose more accuracy than ResNet-style nets.

CIFAR-10 images are upscaled 32→224 so the network keeps its real size; upscaling adds no detail, it only makes the model big.

**Kaggle — before Run All** (right panel): *Accelerator → GPU T4*, *Internet → On* (downloads the pretrained weights),
and *Add Input → CIFAR-10* in the python-pickle format (it must contain a `cifar-10-batches-py` folder). Without it the notebook
downloads CIFAR-10 itself, which took about 50 minutes on a slow Kaggle link before.

**Colab — before Run All:** *Runtime → Change runtime type → T4 GPU*. There is no "Add Input", so CIFAR-10 always downloads (170 MB).
At the end the zip is offered as a browser download. The notebook detects which platform it is on by itself.

Time: the 224×224 training passes dominate. Expect hours, not minutes, on a T4; lower `TRAIN_EPOCHS` / `STEPS` to shorten it
(Kaggle sessions stop after 12 h). Accuracy is scored on a fixed 2000-image slice of the test set, so differences under about 1 pp are noise.

Everything lands in `/kaggle/working/` — download `convnext_tiny_cifar10_compressed.zip` from the *Output* tab (Kaggle) or the browser download prompt (Colab).


In [ ]:
# ======================= CONFIG =======================
ARCH         = "tv:convnext_tiny"  # any torchvision ImageNet model: tv:resnet152, tv:densenet121, tv:convnext_tiny, tv:efficientnet_b0, ...
TRAIN_EPOCHS = 2      # fine-tune the pretrained model on CIFAR-10 (the "original" we compress)
PRUNE_RATIO  = 0.5    # fraction of channels removed (0.5 -> ~75% fewer parameters)
STEPS        = 3      # gradual pruning steps
FT_EPOCHS    = 1      # fine-tune epochs after every step
LOSS         = "dkd"  # "kd" | "dkd"
BATCH        = 32
CALIB        = 256    # calibration images for INT8 (train split)
TEST_IMAGES  = 2000   # test images used to score every stage
# ======================================================

In [ ]:
import os, subprocess, sys, json, shutil, pathlib
IN_COLAB = "COLAB_RELEASE_TAG" in os.environ or os.path.exists("/content")
WORK = "/content" if IN_COLAB and not os.path.exists("/kaggle/working") else "/kaggle/working"
print("platform:", "Colab" if WORK == "/content" else "Kaggle", "| working dir:", WORK)
REPO = "https://github.com/Bhagyesh0603/Model-Compression-ToolKit..git"
if not os.path.exists(f"{WORK}/ck"):
    subprocess.run(["git", "clone", "-q", REPO, f"{WORK}/ck"], check=True)
os.chdir(f"{WORK}/ck")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE -> training would take days on CPU; enable a GPU (Kaggle: right panel; Colab: Runtime > Change runtime type), then Restart & Run All")
from compresskit.zoo.data import resolve_root
print("CIFAR-10 location:", resolve_root(), "(attached dataset if under /kaggle/input, else it will download)")

def run(*args):
    args = [str(a) for a in args]
    print("$", " ".join(args), flush=True)
    return subprocess.run(args).returncode

DATA = ["--dataset", "cifar10-upscaled", "--input-size", "224", "--test-samples", str(TEST_IMAGES)]

## 1 · The original: pretrained ConvNeXt-Tiny with a CIFAR-10 head, fine-tuned

In [ ]:
CKPT = "checkpoints/convnext_tiny_cifar10.pt"
assert run(sys.executable, "-m", "compresskit.cli", "train", "--arch", ARCH, *DATA,
           "--epochs", TRAIN_EPOCHS, "--lr", "0.003", "--batch-size", BATCH, "--out", CKPT) == 0, "training failed - read the log above"

## 2 · Compress: gradual prune + fine-tune → INT8 → evaluate the .onnx

In [ ]:
OUT = WORK
code_ = run(sys.executable, "-m", "compresskit.cli", "pipeline", "--model", CKPT, *DATA,
            "--calib", CALIB, "--batch-size", BATCH,
            "--prune", PRUNE_RATIO, "--steps", STEPS, "--fine-tune-epochs", FT_EPOCHS, "--loss", LOSS,
            "--precision", "int8", "--allow-slow-int8",
            "--workdir", f"{OUT}/work", "--report", f"{OUT}/report.html", "--json", f"{OUT}/results.json")
assert code_ == 0, "pipeline failed - read the log above"

## 3 · Does the compressed model actually work? (original vs ONNX on real test images)

In [ ]:
r = json.load(open(f"{OUT}/results.json"))
shutil.copy(r["stages"][-1]["artifact"], f"{OUT}/model.onnx")
run(sys.executable, "experiments/verify_onnx.py", "--model", CKPT, "--onnx", f"{OUT}/model.onnx",
    "--dataset", "cifar10-upscaled", "--input-size", "224", "--images", 1000)

## 4 · Summary and download bundle

In [ ]:
b, f, t = r["baseline"], r["stages"][-1], r["totals"]
print(f"parameters: {b['parameters']:,} -> {f['parameters']:,}")
print(f"accuracy  : {b['accuracy']*100:.2f}% -> {f['accuracy']*100:.2f}%   (drop {t['accuracy_drop_pp']:.2f} pp, on {TEST_IMAGES} test images)")
print(f"size      : {b['size_mb']:.1f} MB -> {f['size_mb']:.1f} MB   ({t['size_reduction']:.1f}x smaller)")
print(f"precision : {f['precision']}")
print("NOTE: latency in this run is Kaggle-CPU latency, not your device. Measure on the target: compresskit bench model.onnx --input-shape 1 3 224 224")

import zipfile
with zipfile.ZipFile(f"{OUT}/convnext_tiny_cifar10_compressed.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("model.onnx", "report.html", "results.json"):
        z.write(f"{OUT}/{name}", name)
print("bundle:", f"{OUT}/convnext_tiny_cifar10_compressed.zip", f"({os.path.getsize(f'{OUT}/convnext_tiny_cifar10_compressed.zip')/1e6:.1f} MB)")

if WORK == "/content":
    from google.colab import files
    files.download(f"{OUT}/convnext_tiny_cifar10_compressed.zip")


## 5 · On your Mac (real speed check, once)

```bash
unzip convnext_tiny_cifar10_compressed.zip
compresskit bench model.onnx --input-shape 1 3 224 224
```